# Análise de sentimentos

###Testando

In [ ]:
!nvidia-smi

Sat Oct 31 23:16:54 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### Testando o login

In [ ]:
from google.colab import userdata
from huggingface_hub import login

login(userdata.get("HF_TOKEN"))
print("Login no Hugging Face OK")

Login no Hugging Face OK


### Trazer os dados e instalar as bibliotecas

In [ ]:
!git clone https://github.com/laisedosdados/Olist-Data-Ecosystem-Do-ETL-An-lise-de-Sentimentos.git olist
!ls olist/dados

Cloning into 'olist'...
remote: Enumerating objects: 27, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (7/7), done.
remote: Total 27 (delta 2), reused 9 (delta 2), pack-reused 18 (from 2)
Receiving objects: 100% (27/27), 42.31 MiB | 12.52 MiB/s, done.
Resolving deltas: 100% (2/2), done.
olist_customers_dataset.csv	  olist_orders_dataset.csv
olist_geolocation_dataset.csv	  olist_products_dataset.csv
olist_order_items_dataset.csv	  olist_sellers_dataset.csv
olist_order_payments_dataset.csv  product_category_name_translation.csv
olist_order_reviews_dataset.csv


In [ ]:
!pip install -q datasets

### Carregando e limpando

In [ ]:
import pandas as pd

# 1. Lendo somente a tabela de reviews
df = pd.read_csv("olist/dados/olist_order_reviews_dataset.csv")
print(f"Total de reviews: {len(df)}")

# 2. Mantendo apenas reviews com comentário escrito
df = df.dropna(subset=["review_comment_message"])
print(f"Com comentário: {len(df)}")

# 3. Removendo reviews duplicadas
df = df.drop_duplicates(subset="review_id")
print(f"Sem reviews duplicadas: {len(df)}")

# 4. Limpeza mínima: só espaços e quebras de linha extras
df["texto"] = (
    df["review_comment_message"]
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)
df = df[df["texto"].str.len() > 0]

# 5. Criando o rótulo e removendo a nota 3
df = df[df["review_score"] != 3].copy()
df["label"] = (df["review_score"] >= 4).astype(int)  # 1 = positivo, 0 = negativo
print(f"Sem nota 3: {len(df)}")

# 6. Removendo textos idênticos para evitar vazamento entre treino e teste
df = df.drop_duplicates(subset="texto")
print(f"Base final: {len(df)}")
print(df["label"].value_counts(normalize=True).round(3))

Total de reviews: 99224
Com comentário: 40977
Sem reviews duplicadas: 40668
Sem nota 3: 37113
Base final: 32433
label
1    0.68
0    0.32
Name: proportion, dtype: float64


### Dividindo em treino, validação e teste

In [ ]:
from sklearn.model_selection import train_test_split

dados = df[["texto", "label"]]

# Primeiro separa 80% para treino e 20% para o resto
treino, resto = train_test_split(
    dados, test_size=0.2, stratify=dados["label"], random_state=42
)
# Depois divide o resto ao meio: validação e teste
validacao, teste = train_test_split(
    resto, test_size=0.5, stratify=resto["label"], random_state=42
)

for nome, parte in [("Treino", treino), ("Validação", validacao), ("Teste", teste)]:
    print(f"{nome}: {len(parte)} exemplos | {parte['label'].mean():.1%} positivos")

Treino: 25946 exemplos | 68.0% positivos
Validação: 3243 exemplos | 68.0% positivos
Teste: 3244 exemplos | 68.0% positivos


### Olhando exemplos e salvando

In [ ]:
# Olhando alguns exemplos para conferir se os rótulos fazem sentido
print(treino.sample(8, random_state=1).to_string())

# Salvando as divisões para usar nas próximas etapas
import os
os.makedirs("dados_sentimento", exist_ok=True)
treino.to_csv("dados_sentimento/treino.csv", index=False)
validacao.to_csv("dados_sentimento/validacao.csv", index=False)
teste.to_csv("dados_sentimento/teste.csv", index=False)
print("Arquivos salvos!")

                                                                                                                                                                                                          texto  label
1275                                                                                                                                                                    Demorou quase trinta dias .tempo demais      1
5592                                                                                                                                                             É realmente o prometido! Parabéns ao vendedor!      1
65949                                                                                                                                                                        Produto veio certinho e no prazo!!      1
53760                                                                                                                                       

### Treinando o baseline

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, classification_report, confusion_matrix

treino = pd.read_csv("dados_sentimento/treino.csv")
validacao = pd.read_csv("dados_sentimento/validacao.csv")

# Transformando textos em números
vetorizador = TfidfVectorizer(
    ngram_range=(1, 2),   # palavras soltas e pares de palavras
    min_df=2,             # ignora termos que aparecem em só 1 comentário
    max_features=50000,
)
X_treino = vetorizador.fit_transform(treino["texto"])   # aprende o vocabulário só no treino
X_validacao = vetorizador.transform(validacao["texto"])

# Treinando o classificador
baseline = LogisticRegression(max_iter=1000, class_weight="balanced")
baseline.fit(X_treino, treino["label"])

# Avaliando na validação
pred = baseline.predict(X_validacao)
print(f"F1-macro (validação): {f1_score(validacao['label'], pred, average='macro'):.4f}\n")
print(classification_report(validacao["label"], pred,
                            target_names=["negativo", "positivo"], digits=3))
print("Matriz de confusão:")
print(confusion_matrix(validacao["label"], pred))

F1-macro (validação): 0.9124

              precision    recall  f1-score   support

    negativo      0.830     0.947     0.885      1039
    positivo      0.973     0.909     0.940      2204

    accuracy                          0.921      3243
   macro avg      0.902     0.928     0.912      3243
weighted avg      0.927     0.921     0.922      3243

Matriz de confusão:
[[ 984   55]
 [ 201 2003]]


### O que o modelo aprendeu

In [ ]:
import numpy as np

palavras = np.array(vetorizador.get_feature_names_out())
pesos = baseline.coef_[0]
ordem = np.argsort(pesos)

print("Termos mais NEGATIVOS:", list(palavras[ordem[:15]]))
print("Termos mais POSITIVOS:", list(palavras[ordem[-15:]]))

Termos mais NEGATIVOS: ['não', 'nao', 'comprei', 'não recebi', 'não recomendo', 'péssima', 'nem', 'péssimo', 'diferente', 'demora', 'veio', 'aguardando', 'quero', 'ruim', 'quebrado']
Termos mais POSITIVOS: ['antes do', 'lindo', 'amei', 'adorei', 'gostei', 'rápida', 'parabéns', 'rápido', 'antes', 'bem', 'tudo', 'bom', 'ótimo', 'excelente', 'recomendo']


# Fine-tuning

### Vendo a tokenização

In [ ]:
from transformers import AutoTokenizer

NOME_MODELO = "neuralmind/bert-base-portuguese-cased"
tokenizer = AutoTokenizer.from_pretrained(NOME_MODELO)

exemplo = "Produto chegou bem antes do prazo porém está com defeito, vlw"
print(tokenizer.tokenize(exemplo))

['Produ', '##to', 'chegou', 'bem', 'antes', 'do', 'prazo', 'porém', 'está', 'com', 'defe', '##ito', ',', 'v', '##l', '##w']


### Preparando os dados

In [ ]:
import pandas as pd
from datasets import Dataset

treino = pd.read_csv("dados_sentimento/treino.csv")
validacao = pd.read_csv("dados_sentimento/validacao.csv")

ds_treino = Dataset.from_pandas(treino, preserve_index=False)
ds_validacao = Dataset.from_pandas(validacao, preserve_index=False)

def tokenizar(lote):
    return tokenizer(lote["texto"], truncation=True, max_length=128)

ds_treino = ds_treino.map(tokenizar, batched=True)
ds_validacao = ds_validacao.map(tokenizar, batched=True)
print(ds_treino)

Map:   0%|          | 0/25946 [00:00<?, ? examples/s]

Map:   0%|          | 0/3243 [00:00<?, ? examples/s]

Dataset({
    features: ['texto', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 25946
})


### Configurando modelo e treino

In [ ]:
import numpy as np
from transformers import (AutoModelForSequenceClassification, TrainingArguments,
                          Trainer, DataCollatorWithPadding)
from sklearn.metrics import f1_score

# Modelo pré-treinado + cabeça de classificação com 2 classes
modelo = AutoModelForSequenceClassification.from_pretrained(
    NOME_MODELO,
    num_labels=2,
    id2label={0: "negativo", 1: "positivo"},
    label2id={"negativo": 0, "positivo": 1},
)

# Função que calcula o F1-macro a cada avaliação
def calcular_metricas(saida):
    logits, rotulos = saida
    previsoes = np.argmax(logits, axis=-1)
    return {"f1_macro": f1_score(rotulos, previsoes, average="macro")}

args = TrainingArguments(
    output_dir="checkpoints",
    num_train_epochs=2,
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    weight_decay=0.01,
    eval_strategy="epoch",          # avalia ao fim de cada época
    save_strategy="epoch",
    load_best_model_at_end=True,    # no final, fica com a melhor versão
    metric_for_best_model="f1_macro",
    save_total_limit=1,
    fp16=True,
    seed=42,
    report_to="none",
)

trainer = Trainer(
    model=modelo,
    args=args,
    train_dataset=ds_treino,
    eval_dataset=ds_validacao,
    data_collator=DataCollatorWithPadding(tokenizer),  # agrupa textos de tamanhos diferentes
    compute_metrics=calcular_metricas,
)

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  438MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: neuralmind/bert-base-portuguese-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from th

### Treinando o modelo

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,F1 Macro
1,0.204653,0.155100,0.930627
2,0.125715,0.168908,0.934477


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1622, training_loss=0.15474116199554555, metrics={'train_runtime': 231.0574, 'train_samples_per_second': 224.585, 'train_steps_per_second': 7.02, 'total_flos': 1637825208513600.0, 'train_loss': 0.15474116199554555, 'epoch': 2.0})

### Avaliando e salvando

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

saida = trainer.predict(ds_validacao)
pred_bert = np.argmax(saida.predictions, axis=-1)

print(f"F1-macro BERTimbau (validação): {f1_score(validacao['label'], pred_bert, average='macro'):.4f}")
print(f"F1-macro Baseline  (validação): 0.9124\n")
print(classification_report(validacao["label"], pred_bert,
                            target_names=["negativo", "positivo"], digits=3))
print("Matriz de confusão:")
print(confusion_matrix(validacao["label"], pred_bert))

# Salvando o modelo treinado
trainer.save_model("modelo_bertimbau")
tokenizer.save_pretrained("modelo_bertimbau")
print("Modelo salvo!")

F1-macro BERTimbau (validação): 0.9345
F1-macro Baseline  (validação): 0.9124

              precision    recall  f1-score   support

    negativo      0.886     0.940     0.912      1039
    positivo      0.971     0.943     0.957      2204

    accuracy                          0.942      3243
   macro avg      0.928     0.942     0.934      3243
weighted avg      0.944     0.942     0.942      3243

Matriz de confusão:
[[ 977   62]
 [ 126 2078]]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Modelo salvo!


### Subindo pro Hugging Face

In [ ]:
from huggingface_hub import HfApi, whoami

usuario = whoami()["name"]
repo_id = f"{usuario}/bertimbau-sentimento-olist"

api = HfApi()
api.create_repo(repo_id, private=True, exist_ok=True)
api.upload_folder(
    folder_path="modelo_bertimbau",
    repo_id=repo_id,
    commit_message="Primeira versão: BERTimbau ajustado para sentimento em reviews da Olist",
)
print(f"Modelo salvo em: https://huggingface.co/{repo_id}")

KeyboardInterrupt: 

# Avaliação final e análise de erros

### Resultado oficial no teste

In [ ]:
import pandas as pd
import numpy as np
from datasets import Dataset
from scipy.special import softmax
from sklearn.metrics import f1_score, confusion_matrix

teste = pd.read_csv("dados_sentimento/teste.csv")

# Baseline: usa o mesmo vetorizador treinado na Etapa 2
pred_base = baseline.predict(vetorizador.transform(teste["texto"]))

# BERTimbau: tokeniza e prevê
ds_teste = Dataset.from_pandas(teste, preserve_index=False).map(tokenizar, batched=True)
logits = trainer.predict(ds_teste).predictions
probs = softmax(logits, axis=1)   # transforma as pontuações em probabilidades
pred_bert = probs.argmax(axis=1)

for nome, pred in [("Baseline", pred_base), ("BERTimbau", pred_bert)]:
    print(f"===== {nome} =====")
    print(f"F1-macro (teste): {f1_score(teste['label'], pred, average='macro'):.4f}")
    print(confusion_matrix(teste["label"], pred), "\n")

Map:   0%|          | 0/3244 [00:00<?, ? examples/s]

===== Baseline =====
F1-macro (teste): 0.9071
[[ 972   67]
 [ 204 2001]] 

===== BERTimbau =====
F1-macro (teste): 0.9328
[[ 967   72]
 [ 120 2085]] 



###  Dividindo em quatro grupos

In [ ]:
teste["pred_base"] = pred_base
teste["pred_bert"] = pred_bert
teste["confianca_bert"] = probs.max(axis=1)

acerto_base = teste["pred_base"] == teste["label"]
acerto_bert = teste["pred_bert"] == teste["label"]

print("Os dois acertaram:     ", (acerto_base & acerto_bert).sum())
print("Só o BERTimbau acertou:", (~acerto_base & acerto_bert).sum())
print("Só o baseline acertou: ", (acerto_base & ~acerto_bert).sum())
print("Os dois erraram:       ", (~acerto_base & ~acerto_bert).sum())

Os dois acertaram:      2927
Só o BERTimbau acertou: 125
Só o baseline acertou:  46
Os dois erraram:        146


### Testando a hipótese do não

In [ ]:
negacao = teste["texto"].str.contains(
    r"\b(?:não|nao|nem|nada|nenhum|nunca)\b", case=False, regex=True
)
so_bert = ~acerto_base & acerto_bert

print(f"Textos com negação no teste inteiro:            {negacao.mean():.1%}")
print(f"Textos com negação onde só o BERTimbau acertou: {negacao[so_bert].mean():.1%}\n")

print("Exemplos que só o BERTimbau acertou, com negação:")
for _, linha in teste[so_bert & negacao].head(10).iterrows():
    rotulo = "positivo" if linha["label"] == 1 else "negativo"
    print(f"[{rotulo}] {linha['texto']}")

Textos com negação no teste inteiro:            27.2%
Textos com negação onde só o BERTimbau acertou: 54.4%

Exemplos que só o BERTimbau acertou, com negação:
[positivo] Gostei muito. Como não tinham na cor escolhida na venda, mandaram mensagem de Whatsapp perguntando se aceitava outras cores. Foram bem prestativos e respeitosos
[positivo] Recebi produto bm antes do prazo, só que não veio da cor que pedi .
[positivo] Só problema é que não avisa que é preciso ter um cartucho
[positivo] Acabei de receber os dois cartuchos(preto e colorido), porém ainda nao testei
[positivo] No site da lannister a informação do tipo de saco descartável é NANO. No site não diz o número, por isso recebi o NANO 11 que é maior que o saco do aspirador, vou ver se funciona corretamente.
[positivo] Ainda não abri o pacote pois vou dar de prsente de aniversário. Acredito que a minha nora vai gostar muito. É pelo aniversário dela em 13/10. Obrigada
[positivo] Comprei o produto apesar de não ter nenhuma avaliação..

### Onde os dois erraram

In [ ]:
erros = teste[~acerto_base & ~acerto_bert].sort_values("confianca_bert", ascending=False)
print(f"Os dois erraram em {len(erros)} textos. Os 20 em que o BERTimbau estava mais confiante:\n")

for _, linha in erros.head(20).iterrows():
    rotulo = "positivo" if linha["label"] == 1 else "negativo"
    print(f"[rótulo: {rotulo} | confiança: {linha['confianca_bert']:.0%}] {linha['texto']}")

Os dois erraram em 146 textos. Os 20 em que o BERTimbau estava mais confiante:

[rótulo: negativo | confiança: 100%] Boneca linda!!! e a entrega no prazo , muito satisfeita .
[rótulo: negativo | confiança: 100%] recomendo demais
[rótulo: negativo | confiança: 100%] deu tudo certo gostei
[rótulo: negativo | confiança: 100%] Loja razuavel
[rótulo: negativo | confiança: 100%] Bom produto, recomendo o vendedor. Mais nada pra comentar do produto e do vendedor também.
[rótulo: negativo | confiança: 100%] Produto otimo so demorou chegar...
[rótulo: negativo | confiança: 99%] Capinha meia boca, mas atende o proposto!
[rótulo: negativo | confiança: 99%] Foi entregue na data,só acho que a taxa de entrega poderia ser mais em conta.
[rótulo: negativo | confiança: 99%] Dentro do prazo previsto.
[rótulo: positivo | confiança: 99%] Não recebi o produto, enviei mensagem não me responderam, liguei estou esperando até agora o retorno
[rótulo: positivo | confiança: 99%] Nao recebi o produto,ja liguei var

### Recarregando tudo

In [1]:
# ===== RECARREGAR TUDO =====
!pip install -q datasets
!test -d olist || git clone -q https://github.com/laisedosdados/Olist-Data-Ecosystem-Do-ETL-An-lise-de-Sentimentos.git olist

import os, numpy as np, pandas as pd, torch
from google.colab import userdata
from huggingface_hub import login, whoami
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, confusion_matrix
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# 1. Login no Hugging Face
login(userdata.get("HF_TOKEN"))
repo_id = f"{whoami()['name']}/bertimbau-sentimento-olist"

# 2. Recriar a base e a divisão (random_state=42 gera exatamente a mesma divisão)
df = pd.read_csv("olist/dados/olist_order_reviews_dataset.csv")
df = df.dropna(subset=["review_comment_message"]).drop_duplicates(subset="review_id")
df["texto"] = df["review_comment_message"].str.replace(r"\s+", " ", regex=True).str.strip()
df = df[(df["texto"].str.len() > 0) & (df["review_score"] != 3)].copy()
df["label"] = (df["review_score"] >= 4).astype(int)
df = df.drop_duplicates(subset="texto")
dados = df[["texto", "label"]]
treino, resto = train_test_split(dados, test_size=0.2, stratify=dados["label"], random_state=42)
validacao, teste = train_test_split(resto, test_size=0.5, stratify=resto["label"], random_state=42)
teste = teste.reset_index(drop=True)
os.makedirs("dados_sentimento", exist_ok=True)
treino.to_csv("dados_sentimento/treino.csv", index=False)
validacao.to_csv("dados_sentimento/validacao.csv", index=False)
teste.to_csv("dados_sentimento/teste.csv", index=False)

# 3. Retreinar o baseline (leva 1 segundo)
vetorizador = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=50000)
baseline = LogisticRegression(max_iter=1000, class_weight="balanced")
baseline.fit(vetorizador.fit_transform(treino["texto"]), treino["label"])
pred_base = baseline.predict(vetorizador.transform(teste["texto"]))

# 4. Carregar o BERTimbau já treinado, direto do seu Hugging Face
device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained(repo_id)
modelo = AutoModelForSequenceClassification.from_pretrained(repo_id).to(device).eval()

probs = []
with torch.no_grad():
    for i in range(0, len(teste), 64):
        lote = tokenizer(teste["texto"][i:i+64].tolist(), truncation=True,
                         max_length=128, padding=True, return_tensors="pt").to(device)
        probs.append(torch.softmax(modelo(**lote).logits, dim=-1).cpu().numpy())
probs = np.vstack(probs)
pred_bert = probs.argmax(axis=1)

# 5. Conferência
print(f"Treino: {len(treino)} | Validação: {len(validacao)} | Teste: {len(teste)}")
print(f"Baseline  F1-macro (teste): {f1_score(teste['label'], pred_base, average='macro'):.4f}")
print(f"BERTimbau F1-macro (teste): {f1_score(teste['label'], pred_bert, average='macro'):.4f}")
print(f"Rodando em: {device}")

config.json:   0%|          | 0.00/1.07k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/380 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/678k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  436MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Treino: 25946 | Validação: 3243 | Teste: 3244
Baseline  F1-macro (teste): 0.9071
BERTimbau F1-macro (teste): 0.9328
Rodando em: cuda


# Etapa 5

### Subindo o model card

In [2]:
from huggingface_hub import HfApi

card = """---
language: pt
license: cc-by-nc-sa-4.0
base_model: neuralmind/bert-base-portuguese-cased
pipeline_tag: text-classification
tags:
  - sentiment-analysis
  - portuguese
  - e-commerce
  - bertimbau
metrics:
  - f1
---

# BERTimbau: sentimento em reviews de e-commerce

Classifica comentários de clientes em português como **positivo** ou **negativo**.
É o BERTimbau Base ajustado (fine-tuning) com cerca de 26 mil reviews reais do e-commerce brasileiro da Olist.

## Como usar

```python
from transformers import pipeline

classificador = pipeline("text-classification", model="REPO_ID")
classificador("Chegou antes do prazo, mas veio com defeito.")
```

## Dados

- **Origem:** Brazilian E-Commerce Public Dataset by Olist (reviews de 2016 a 2018)
- **Rótulo:** derivado da nota do cliente: 1–2 = negativo, 4–5 = positivo; a nota 3 foi excluída por ser ambígua
- **Limpeza:** remoção de reviews sem comentário, de reviews duplicadas e de textos idênticos (para evitar vazamento entre treino e teste); stopwords mantidas, porque palavras como "não" carregam o sentido
- **Base final:** 32.433 comentários (68% positivos), divididos em 80% treino, 10% validação e 10% teste, de forma estratificada

## Resultados (conjunto de teste, 3.244 comentários)

| Modelo | F1-macro | Acurácia | Erros |
|---|---|---|---|
| TF-IDF + Regressão Logística (baseline) | 0,907 | 91,6% | 271 |
| **Este modelo** | **0,933** | **94,1%** | **192** |

Na classe negativa, a precisão subiu de 82,7% para 89,0%, com recall praticamente igual (93,6% e 93,1%).
O ganho veio principalmente da redução de comentários positivos classificados como negativos (de 204 para 120).

## Análise de erros

- **Negação:** palavras como "não" e "nada" aparecem em 27% dos comentários de teste, mas em 54% dos casos que só este modelo acertou. Ele entende frases como "ainda não testei, mas parece ótimo", que o baseline classifica como negativas.
- **Ruído nos rótulos:** entre os 20 erros mais confiantes cometidos pelos dois modelos, 13 tinham rótulo inconsistente com o texto (por exemplo, "não recebi o produto" com nota alta) e 7 tinham sentimento misto. Por isso, parte do erro medido vem do dado, e não do modelo.
- **Entrega pesa na nota:** em vários casos mistos, o cliente elogia o produto e dá nota baixa por causa da demora ou do frete.

## Limitações

- O rótulo vem da nota, e não de anotação humana do texto, então contém ruído.
- Não há classe neutra; comentários mistos são forçados para positivo ou negativo.
- Treinado só com e-commerce brasileiro de 2016 a 2018, com comentários curtos (mediana de 9 palavras). Não foi avaliado em outros domínios.
- Nomes de lojas na base original foram anonimizados.

## Treinamento

2 épocas, learning rate 2e-5, batch size 32, weight decay 0,01, comprimento máximo de 128 tokens, fp16, seed 42, GPU T4 (Google Colab).
Foi mantida a versão com melhor F1-macro na validação.

## Código

Pipeline completo, do ETL à análise de erros: https://github.com/laisedosdados/Olist-Data-Ecosystem-Do-ETL-An-lise-de-Sentimentos

**Autora:** Laíse Soares Alves
"""

card = card.replace("REPO_ID", repo_id)
api = HfApi()
api.upload_file(
    path_or_fileobj=card.encode("utf-8"),
    path_in_repo="README.md",
    repo_id=repo_id,
    commit_message="Adiciona model card",
)
print(f"Card publicado! Revise em: https://huggingface.co/{repo_id}")

Card publicado! Revise em: https://huggingface.co/laisedosdados/bertimbau-sentimento-olist


### Testando o modelo

In [3]:
from transformers import pipeline

classificador = pipeline("text-classification", model=repo_id,
                         device=0 if torch.cuda.is_available() else -1)

frases = [
    "Chegou antes do prazo, mas veio com defeito.",
    "Ainda não testei, mas parece ótimo.",
    "Não recebi o produto até hoje.",
    "Não tenho nada a reclamar, recomendo!",
    "Produto bom, porém a entrega demorou demais.",
]
for frase, resultado in zip(frases, classificador(frases)):
    print(f"{resultado['label']:>8} ({resultado['score']:.0%})  {frase}")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

negativo (94%)  Chegou antes do prazo, mas veio com defeito.
positivo (100%)  Ainda não testei, mas parece ótimo.
negativo (99%)  Não recebi o produto até hoje.
positivo (100%)  Não tenho nada a reclamar, recomendo!
positivo (57%)  Produto bom, porém a entrega demorou demais.


### Colocando como público

In [4]:
api.update_repo_settings(repo_id, private=False)
print(f"Modelo público em: https://huggingface.co/{repo_id}")

Modelo público em: https://huggingface.co/laisedosdados/bertimbau-sentimento-olist


## Baixando o notebook

In [ ]:
import json
from google.colab import _message, files

nb = _message.blocking_request("get_ipynb")["ipynb"]

# Remove os metadados de widgets que quebram a visualização no GitHub
nb.get("metadata", {}).pop("widgets", None)
for celula in nb["cells"]:
    celula.get("metadata", {}).pop("widgets", None)

# Segurança: tokens do Hugging Face começam com "hf_"
texto = json.dumps(nb)
print(" Possível token no notebook!" if "hf_" in texto else "Nenhum token encontrado ")

with open("02_analise_de_sentimentos.ipynb", "w", encoding="utf-8") as f:
    json.dump(nb, f, ensure_ascii=False, indent=1)
files.download("02_analise_de_sentimentos.ipynb")